In [ ]:
import os
import pathlib
import sys

# 노트북이 어느 위치에서 실행되든 backend/app 이 들어 있는 폴더(hanwha-agent)를 찾아 루트로 삼는다
# - 폴더 이름(day05)에 기대지 않으므로 다른 날짜 노트북에 복사해도 그대로 쓸 수 있다
here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)                                  # 상대경로(.env, app.db 등)의 기준
SANDBOX = ROOT / "sandbox" / "w3" / "day05"

# app 패키지를 import 할 수 있게 backend 를 모듈 검색 경로 맨 앞에 넣는다
# - os.chdir 만으로는 import 경로가 바뀌지 않는다
BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

print("프로젝트 루트  :", ROOT)


- 가드 함수 검사

In [ ]:
import importlib
importlib.invalidate_caches() 

from app.core.exceptions import GuardTripped, RateLimited    
from app.core.guards import ALLOWED_MODELS, check_daily_limit, check_model, check_question
from app.core.config import get_settings

settings = get_settings()

def try_guard(label: str, func, arg) -> None:

    try:
        result = func(arg)
    except (GuardTripped, RateLimited) as e:
        code = getattr(e, "status_code", None)
        print(f"{label} : {type(e).__name__} ({code}) {e}")
    else:
        print(f'{label} : 통과 → "{result}"')


print("허용 모델      :", ALLOWED_MODELS)     
print()

try_guard("정상 질문     ", check_question, "부산 출장 숙박비 한도가 얼마인가요?")
try_guard("공백 세 칸    ", check_question, "   ")
try_guard("아주 긴 질문  ", check_question, "가" * (settings.max_input_chars + 1))
try_guard("허용 밖 모델  ", check_model, "claude-opus-4-1")
try_guard("한도 초과     ", check_daily_limit, settings.daily_call_limit)

ModuleNotFoundError: No module named 'app'

In [ ]:
from pydantic import ValidationError
from app.schemas.chat import AnswerOut

CASES = [
    ("ⓐ 필드 누락  ", '{"answer": "1박 70,000원 이내입니다.", "enough_evidence": true}'),
    ("ⓑ 타입 어긋남", '{"answer": "1박 70,000원 이내입니다.", "sources": "DOC-HR-014", "enough_evidence": true}'),
    ("ⓒ 잡담이 앞에", '네, 알겠습니다!\n{"answer": "1박 70,000원 이내입니다.", "sources": [], "enough_evidence": false}'),
]

for label, raw in CASES:
    try:
        AnswerOut.model_validate_json(raw)
        print(f"{label} : 통과")
    except ValidationError as e:
        first = e.errors(include_url=False)[0]
        print(f"{label} : loc={str(first['loc'])} / type={first['type']} / {first['msg']}")

In [ ]:
import importlib
import app.schemas.chat as chat_schema

chat_schema = importlib.reload(chat_schema)

print('AskOut 필드 :', list(chat_schema.AskOut.model_fields))

sample = chat_schema.AskOut(run_id='RUN-8821', answer='1박 70,000원 이내입니다.', sources=[], enough_evidence=False)
print(f'기본값      : attempts={sample.attempts} · fallback_used={sample.fallback_used}')

In [1]:
import importlib
import inspect

importlib.invalidate_caches()    

from app.services import chat_service

print("ask 시그니처 :", inspect.signature(chat_service.ask, eval_str=True))
print(f"MAX_ATTEMPTS : {chat_service.MAX_ATTEMPTS}   (첫 호출 1 + 재시도 2)")


ask_body = inspect.getsource(chat_service.ask)

caught_names = []                                 
for line in ask_body.splitlines():
    stripped = line.strip()                     
    if not stripped.startswith("except "):
        continue                                

    after_except = stripped[len("except "):]      
    exception_name = after_except.split(" as ")[0]  
    caught_names.append(exception_name.rstrip(":")) 

if len(caught_names) == 1:
    how_many = "하나"
else:
    how_many = f"{len(caught_names)}개"
print("잡는 예외    :", " · ".join(caught_names), how_many)

ModuleNotFoundError: No module named 'app'

In [2]:
import json
# 잘된 응답 
GOOD = json.dumps(
    {
        "answer": "국내출장 여비 규정 제12조에 따르면 숙박비는 1박 70,000원 이내입니다.",
        "sources": [
            {
                "doc_id": "DOC-HR-014",
                "title": "국내출장 여비 규정",
                "version": "v2.0",
                "locator": "제12조",
            }
        ],
        "enough_evidence": True,
    },
    ensure_ascii=False,
)
# 잘못된 응답 
NO_SOURCES = '{"answer": "1박 70,000원 이내입니다.", "enough_evidence": true}'
CHITCHAT = '죄송합니다. 지금은 답변을 드릴 수 없습니다.'

def next_answer(replies: list[str], attempt: int) -> str:
    if attempt <= len(replies):
        return replies[attempt - 1]
    return replies[-1]
print('준비한 응답 :', 'GOOD · NO_SOURCES · CHITCHAT 세 벌')


준비한 응답 : GOOD · NO_SOURCES · CHITCHAT 세 벌


In [3]:
from pydantic import ValidationError
from app.integrations.llm_claude import _extract_json
from app.schemas.chat import AnswerOut, AskOut
from app.services.chat_service import MAX_ATTEMPTS, _fallback, _hint_from

def run_loop(replies: list[str], run_id: str='RUN-0000') -> AskOut:
    hint = ''
    for attempt in range(1, MAX_ATTEMPTS + 1):
        raw = next_answer(replies, attempt)
        try:
            data = _extract_json(raw)
            AnswerOut.model_validate(data)
        except ValidationError as exc:
            hint = _hint_from(exc.errors(include_url=False))
            continue
        return AskOut(**data, run_id=run_id, attempts=attempt, fallback_used=False)
    return _fallback(run_id, MAX_ATTEMPTS)
CASES_RETRY = [('① 한 번에 성공  ', [GOOD]), ('② 한 번 실패    ', [NO_SOURCES, GOOD]), ('③ 전부 실패     ', [CHITCHAT, CHITCHAT, CHITCHAT])]
for (label, replies) in CASES_RETRY:
    out = run_loop(replies)
    source_count = len(out.sources)
    print(f'{label}{out.run_id}  attempts={out.attempts}  fallback={str(out.fallback_used):<6} 근거 {source_count}건')
    if out.fallback_used:
        print(f'   ⚠️ 폴백으로 응답합니다 (attempts={out.attempts})')


ModuleNotFoundError: No module named 'app'

- 골든셋 : 문항 하나의 모양

In [ ]:
# 문항은 "정답 문장"이 아니라 "지켜야 할 조건"으로 적는다
# - LLM 은 같은 질문에 매번 다르게 답하므로 문장 비교로는 회귀를 잡을 수 없다
CASE_N01 = {
    "id": "N-01",
    "kind": "N",                          # N 정상 / E 근거없음 / F 스키마위반 / G 가드
    "question": "부산 출장 2박 3일인데 숙박비 한도가 얼마인가요?",
    "stub": [                             # 모델이 이렇게 답한다고 가정한다
        '{"answer": "부산 출장 숙박비는 1박 7만원 이내입니다.", '
        '"sources": [{"doc_id": "DOC-HR-014", "title": "국내출장 여비 규정", '
        '"version": "v2.0", "locator": "제12조(숙박비) · p.6"}], '
        '"enough_evidence": true}'
    ],
    "expect": {
        "contains": ["7만"],              # 핵심 숫자 하나가 답에 있어야 한다
        "min_sources": 1,                 # 근거가 1건 이상
        "doc_ids": ["DOC-HR-014"],        # 이 문서에서 나와야 한다
    },
}

CASE_E01 = {
    "id": "E-01",
    "kind": "E",
    "question": "화성 출장 숙박비 한도는 얼마인가요?",
    "stub": [
        '{"answer": "국내출장 여비 규정에는 해당 지역의 숙박비 한도가 없습니다. '
        '담당 부서에 문의해 주세요.", "sources": [], "enough_evidence": false}'
    ],
    "expect": {
        "max_sources": 0,                 # 근거를 달면 안 된다
        "enough_evidence": False,         # 스스로 근거가 부족하다고 말해야 한다
        "not_contains": ["7만"],          # 다른 지역 숫자를 끌어다 지어내면 실패
    },
}

print("N-01 expect 키 :", list(CASE_N01["expect"].keys()))
print("E-01 expect 키 :", list(CASE_E01["expect"].keys()))

In [ ]:
import json
from collections import Counter

# 문항은 코드가 아니라 파일에 둔다 -> 문항을 늘려도 테스트 코드를 고치지 않는다
GOLDEN_PATH = ROOT / "backend" / "tests" / "golden" / "chat_golden.json"
GOLDEN = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

counts = Counter(case["kind"] for case in GOLDEN)

KIND_LABELS = {
    "N": "N (정상)       ",
    "E": "E (근거 없음)  ",
    "F": "F (스키마 위반)",
    "G": "G (가드)       ",
}
print(f"문항 {len(GOLDEN)}개")
for kind, label in KIND_LABELS.items():
    print(f"  {label} {counts[kind]}")

# 한 유형이라도 0 이면 그 상황은 아예 검사되지 않고 있다는 뜻이다
print("네 유형 모두 있는가 :", all(counts[kind] > 0 for kind in KIND_LABELS))

- 가짜 어댑터로 갈아끼우기

In [ ]:
import app.integrations.factory as factory
from app.integrations.ports import LLMResult
from app.services import chat_service

# 미리 정해둔 응답을 순서대로 돌려주는 가짜 어댑터
# - LLMPort 를 상속하지 않는다. Protocol 이라 answer 메서드만 맞으면 그대로 들어간다
# - 키워드 전용(*)까지 포트와 똑같이 맞춘다. Protocol 은 이름만 보고 서명은 검사하지 않아서
#   여기서 어긋나면 isinstance 는 통과하는데 호출에서 TypeError 가 난다
class StubLLM:
    name = "stub"

    def __init__(self, replies: list[str]) -> None:
        self.replies = list(replies)
        self.calls = 0

    def answer(self, *, question: str, contexts: list[dict], user: dict) -> LLMResult:
        # 응답이 더 필요해지면 마지막 것을 계속 돌려준다
        text = self.replies[min(self.calls, len(self.replies) - 1)]
        self.calls += 1
        return LLMResult(
            text=text,
            model="claude-haiku-4-5",
            input_tok=1200,
            output_tok=300,
            cost_krw=2.3,
            latency_ms=900,
        )

REAL_LLM = factory.get_llm          # 원래 함수를 먼저 챙겨둔다

stub = StubLLM([
    '{"answer": "부산 출장 숙박비는 1박 7만원 입니다.", '
    '"sources": [{"doc_id": "DOC-HR-014", "title": "국내출장 여비 규정", '
    '"version": "v2.0", "locator": "제12조(숙박비) · p.6"}], '
    '"enough_evidence": true}'
])

# 팩토리의 get_llm 만 갈아끼운다 -> 서비스 코드는 한 줄도 손대지 않는다
# - chat_service 가 factory 모듈을 통째로 import 해뒀기 때문에 이 교체가 먹는다
#   (from ... import get_llm 이었다면 서비스가 쥔 이름은 안 바뀌어 교체가 안 먹는다)
factory.get_llm = lambda: stub
try:
    out = chat_service.ask(question="부산 출장 2박 3일인데 숙박비 한도가 얼마인가요?")
finally:
    factory.get_llm = REAL_LLM      # 되돌리지 않으면 이후 셀이 계속 가짜를 쓴다

print("answer        :", out.answer)
print("attempts      :", out.attempts)
print("stub 호출 횟수 :", stub.calls)
print("fallback_used :", out.fallback_used)

- 판정 함수

In [ ]:
from types import SimpleNamespace

# 판정 결과를 assert 가 아니라 "실패 사유 목록"으로 돌려준다
# - assert 면 첫 위반에서 멈춘다. 목록이면 어긋난 것을 한 번에 모두 볼 수 있다
def check(case: dict, out) -> list[str]:
    expect = case["expect"]
    problems: list[str] = []

    # 핵심 숫자·낱말이 답에 있는지 (문장 전체가 아니라 요점만 본다)
    for needle in expect.get("contains", []):
        if needle not in out.answer:
            problems.append(f"answer 에 '{needle}' 이 없습니다")

    # 근거에 없는 내용을 끌어다 지어내지 않았는지
    for needle in expect.get("not_contains", []):
        if needle in out.answer:
            problems.append(f"answer 에 '{needle}' 이 들어 있습니다")

    # 근거 건수 — 있어야 할 때는 최소, 없어야 할 때는 최대로 본다
    if "min_sources" in expect and len(out.sources) < expect["min_sources"]:
        problems.append(
            f"근거가 {expect['min_sources']}건 이상이어야 합니다 (현재 {len(out.sources)}건)"
        )
    if "max_sources" in expect and len(out.sources) > expect["max_sources"]:
        problems.append(
            f"sources 가 {expect['max_sources']}건이어야 합니다 (현재 {len(out.sources)}건)"
        )

    # 실행 정보 — 몇 번 만에 성공했는지, 폴백이었는지, 근거가 충분하다고 했는지
    for key in ("attempts", "fallback_used", "enough_evidence"):
        if key in expect and getattr(out, key) != expect[key]:
            problems.append(f"{key} 가 {expect[key]} 여야 합니다")

    # 화이트리스트 — 등록된 문서 번호만 인용했는지
    if "doc_ids" in expect:
        for source in out.sources:
            if source.doc_id not in expect["doc_ids"]:
                problems.append(f"허용되지 않은 doc_id: {source.doc_id}")

    return problems

# 판정 함수만 따로 확인하려고 만든 가짜 응답 객체
# - AskOut 을 만들면 필드를 다 채워야 해서, 판정 로직만 볼 때는 SimpleNamespace 가 가볍다
def fake_out(answer: str, doc_ids: list[str], **fields) -> SimpleNamespace:
    base = {"enough_evidence": True, "attempts": 1, "fallback_used": False}
    base.update(fields)
    return SimpleNamespace(
        answer=answer,
        sources=[SimpleNamespace(doc_id=doc_id) for doc_id in doc_ids],
        **base,
    )

In [ ]:
# 판정 함수가 "잡아내야 할 것을 잡는지" 먼저 확인한다
out_ok = fake_out("부산 출장 숙박비는 1박 7만원 이내입니다.", ["DOC-HR-014"])
out_no_number = fake_out("부산 출장 숙박비는 규정에 정해진 한도를 따릅니다.", ["DOC-HR-014"])
out_fake_doc = fake_out("부산 출장 숙박비는 1박 7만원 이내입니다.", ["DOC-HR-999"])

print("정상 응답        :", json.dumps(check(CASE_N01, out_ok), ensure_ascii=False))
print("숫자가 빠짐      :", json.dumps(check(CASE_N01, out_no_number), ensure_ascii=False))
print("없는 문서를 인용 :", json.dumps(check(CASE_N01, out_fake_doc), ensure_ascii=False))

- 통과율

In [ ]:
import logging

from app.core.exceptions import GuardTripped
from app.core.guards import check_model

# 2000자 초과 같은 긴 질문을 JSON 에 그대로 적지 않기 위한 장치
def question_of(case: dict) -> str:
    return case["question"] * case.get("repeat", 1)

# 문항 하나를 돌려 실패 사유 목록을 돌려준다 (빈 목록 = 통과)
def run_case(case: dict) -> list[str]:
    expect = case["expect"]

    # G 유형 — 가드가 막아야 하는 문항. 답이 아니라 "예외가 났는가"를 본다
    # - 여기서 폴백 답변이 나오면 가드를 둔 의미가 없다
    if "raises" in expect:
        try:
            if case.get("guard") == "check_model":
                check_model(case["question"])
            else:
                chat_service.ask(question=question_of(case))
        except GuardTripped as exc:
            return [f"예외 메시지에 '{n}' 이 없습니다"
                    for n in expect.get("contains", []) if n not in str(exc)]
        return ["GuardTripped 가 나지 않았습니다"]

    stub = StubLLM(case["stub"])
    factory.get_llm = lambda: stub
    try:
        out = chat_service.ask(question=question_of(case))
    finally:
        factory.get_llm = REAL_LLM
    # 교체가 실제로 먹었는지 확인한다. 0 이면 판정이 다 통과해도 의미가 없다
    if stub.calls == 0:
        return ["가짜 어댑터가 한 번도 불리지 않았습니다"]
    return check(case, out)

In [ ]:
# F 유형은 일부러 스키마를 어기게 만든 문항이라 재시도 경고가 쏟아진다
# - 통과율 표를 읽으려고 잠시만 끈다. 끝나면 반드시 되돌린다
logging.disable(logging.WARNING)
try:
    results = {case["id"]: run_case(case) for case in GOLDEN}
finally:
    logging.disable(logging.NOTSET)

for kind in KIND_LABELS:
    line = [f"{case['id']} {'YES' if not results[case['id']] else 'NO'}"
            for case in GOLDEN if case["kind"] == kind]
    print("   ".join(line))

passed = sum(1 for problems in results.values() if not problems)
print()
print(f"통과율 : {passed}/{len(GOLDEN)} = {passed / len(GOLDEN) * 100:.1f}%")

# 통과 못한 문항은 사유를 함께 낸다 — 숫자만 보면 무엇을 고쳐야 할지 알 수 없다
for case_id, problems in results.items():
    if problems:
        print(f"  {case_id} : {json.dumps(problems, ensure_ascii=False)}")